# Baseline Recommender 
This notebook builds the first CalCourse recommendation baseline using TF-IDF and cosine similarity to rank eligible courses by relevance to a student profile.

In [1]:
import pandas as pd 
import networkx as nx 

courses = pd.read_csv(
    "../data/processed/recommendable_courses_fall_2026.csv"
)

prereqs = pd.read_csv(
    "../data/processed/prerequisites_fall_2026.csv"
)

In [2]:
prereqs["course"] = (
    prereqs["subject"] + " " + prereqs["course_number"].astype(str)
)

prereqs["prerequisite"] = (
    prereqs["prereq_subject"] + " " + prereqs["prereq_number"].astype(str)
)

G = nx.DiGraph()

for _, row in prereqs.iterrows():
    G.add_edge(row["prerequisite"], row["course"])

In [3]:
completed_courses = {
    "DATA C8",
    "COMPSCI 61A",
    "MATH 1A",
    "MATH 1B"
}

student_interests = """
machine learning statistics data science product analytics
"""

In [4]:
def get_prerequisites(course):
    if course not in G:
        return set()

    return set(G.predecessors(course))


def missing_prerequisites(course, completed_courses):
    return get_prerequisites(course) - completed_courses

In [5]:
courses["course"] = (
    courses["subject"] + " " + courses["course_number"].astype(str)
)

courses["missing_prereqs"] = courses["course"].apply(
    lambda course: missing_prerequisites(course, completed_courses)
)

eligible_courses = courses[
    courses["missing_prereqs"].apply(len) == 0
].copy()

eligible_courses.shape

(1725, 10)

In [6]:
eligible_courses["text"] = (
    eligible_courses["title"].fillna("") + " " +
    eligible_courses["description"].fillna("")
)

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
vectorizer = TfidfVectorizer(
    stop_words="english"
)

In [10]:
course_matrix = vectorizer.fit_transform(
    eligible_courses["text"]
)

In [11]:
student_vector = vectorizer.transform(
    [student_interests]
)

In [12]:
similarity_scores = cosine_similarity(
    student_vector,
    course_matrix
).flatten()

In [13]:
eligible_courses["similarity_score"] = similarity_scores

In [14]:
recommendations = eligible_courses.sort_values(
    "similarity_score",
    ascending=False
)

In [15]:
recommendations[
    ["course", "title", "similarity_score"]
].head(15)

,course,title,similarity_score
1033,INDENG 142A,Introduction to Machine Learning and Data Anal...,0.382353
453,DATA 188,Advanced Data Science Connector,0.302737
439,CYPLAN 101,Introduction to Urban Data Analytics,0.297465
460,STS C104D,Human Contexts and Ethics of Data - DATA/Histo...,0.292148
1916,STAT 157,Seminar on Topics in Probability and Statistics,0.287889
454,DATA 36,Data Scholars Seminar,0.275175
1918,STAT 197,Field Study in Statistics,0.261171
434,STAT C8,Foundations of Data Science,0.210679
464,DATA C6,Introduction to Computational Thinking with Da...,0.203917
458,DATA 94,Special Topics in Data Science,0.202272


### Baseline Observations
The TF-IDF baseline produces some relevant results but tends to over-rank courses with strong keyword overlap, including seminars and special topics. It doesn't capture deeper semantic relevance or distinguish foundational courses from niche offerings. 

## 2. Semantic Recommender
This section tests a semantic embedding model to capture simliarity in meaning rather than exact wording. 

In [16]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [17]:
eligible_courses["text"] = (
    eligible_courses["title"].fillna("") + ". " +
    eligible_courses["description"].fillna("")
)

In [18]:
course_embeddings = model.encode(
    eligible_courses["text"].tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/54 [00:00<?, ?it/s]

In [19]:
student_embedding = model.encode(
    [student_interests]
)

In [20]:
semantic_scores = cosine_similarity(
    student_embedding,
    course_embeddings
).flatten()

In [21]:
eligible_courses["semantic_score"] = semantic_scores

semantic_recommendations = eligible_courses.sort_values(
    "semantic_score",
    ascending=False
)

In [22]:
semantic_recommendations[
    ["course", "title", "semantic_score"]
].head(15)

,course,title,semantic_score
1033,INDENG 142A,Introduction to Machine Learning and Data Anal...,0.493830
1973,UGBA 104,Introduction to Business Analytics,0.446975
596,ENGIN 183D,Product Management,0.430856
1917,STAT 159,Reproducible and Collaborative Statistical Dat...,0.411724
1908,STAT 133,Concepts in Computing with Data,0.357000
219,CDSS 170,Data Discovery Research Practicum,0.323936
453,DATA 188,Advanced Data Science Connector,0.315442
2002,UGBA 162,Brand Management and Strategy,0.304495
253,CHEM 98,Supervised Group Study,0.277914
434,STAT C8,Foundations of Data Science,0.273818


## 3. Baseline Comparison
The semantic recommmender is compared against the TF-IDF baseline to see whether it produces more relevant recommendations for the same student profile. 

In [30]:
tfidf_top10 = recommendations[
    ["course", "title", "similarity_score"]
].head(10)

semantic_top10 = semantic_recommendations[
    ["course", "title", "semantic_score"]
].head(10)

display(tfidf_top10)
display(semantic_top10)

,course,title,similarity_score
1033,INDENG 142A,Introduction to Machine Learning and Data Anal...,0.382353
453,DATA 188,Advanced Data Science Connector,0.302737
439,CYPLAN 101,Introduction to Urban Data Analytics,0.297465
460,STS C104D,Human Contexts and Ethics of Data - DATA/Histo...,0.292148
1916,STAT 157,Seminar on Topics in Probability and Statistics,0.287889
454,DATA 36,Data Scholars Seminar,0.275175
1918,STAT 197,Field Study in Statistics,0.261171
434,STAT C8,Foundations of Data Science,0.210679
464,DATA C6,Introduction to Computational Thinking with Da...,0.203917
458,DATA 94,Special Topics in Data Science,0.202272


,course,title,semantic_score
1033,INDENG 142A,Introduction to Machine Learning and Data Anal...,0.493830
1973,UGBA 104,Introduction to Business Analytics,0.446975
596,ENGIN 183D,Product Management,0.430856
1917,STAT 159,Reproducible and Collaborative Statistical Dat...,0.411724
1908,STAT 133,Concepts in Computing with Data,0.357000
219,CDSS 170,Data Discovery Research Practicum,0.323936
453,DATA 188,Advanced Data Science Connector,0.315442
2002,UGBA 162,Brand Management and Strategy,0.304495
253,CHEM 98,Supervised Group Study,0.277914
434,STAT C8,Foundations of Data Science,0.273818


### Comparison Observations
The semantic recommender produces more coherent reesults than the TF-IDF baseline for the same student proile. TF-IDF tends to favor direct keyword overlap while semantic embeddings surface conceptually related courses in areas such as business analytics, product management, market research, and statistical computing. 

Some weak recommendations remain, which suggests that semantic similarity alone is not sufficient for final ranking.

## 3. Hybrid Ranking

In [32]:
student_profile = {
    "interests": [
        "machine learning",
        "statistics",
        "data science",
        "product analytics"
    ],
    "career_goal": "product manager",
    "preferred_subjects": [
        "DATA",
        "STAT",
        "COMPSCI",
        "UGBA"
    ]
}

eligible_courses["subject_fit"] = eligible_courses["subject"].apply(
    lambda x: 1 if x in student_profile["preferred_subjects"] else 0
)

special_pattern = (
    "special study|group study|field study|independent study|"
    "research|thesis"
)

eligible_courses["special_penalty"] = eligible_courses["title"].str.contains(
    special_pattern,
    case=False,
    na=False
).astype(int)

eligible_courses["final_score"] = (
    0.75 * eligible_courses["semantic_score"]
    + 0.15 * eligible_courses["subject_fit"]
    - 0.10 * eligible_courses["special_penalty"]
)

hybrid_recommendations = eligible_courses.sort_values(
    "final_score",
    ascending=False
)

hybrid_recommendations[
    ["course", "title", "semantic_score", "subject_fit", "special_penalty", "final_score"]
].head(15)

,course,title,semantic_score,subject_fit,special_penalty,final_score
1973,UGBA 104,Introduction to Business Analytics,0.446975,1,0,0.485231
1917,STAT 159,Reproducible and Collaborative Statistical Dat...,0.411724,1,0,0.458793
1908,STAT 133,Concepts in Computing with Data,0.357000,1,0,0.417750
453,DATA 188,Advanced Data Science Connector,0.315442,1,0,0.386582
2002,UGBA 162,Brand Management and Strategy,0.304495,1,0,0.378371
1033,INDENG 142A,Introduction to Machine Learning and Data Anal...,0.493830,0,0,0.370372
434,STAT C8,Foundations of Data Science,0.273818,1,0,0.355364
458,DATA 94,Special Topics in Data Science,0.264483,1,0,0.348362
464,DATA C6,Introduction to Computational Thinking with Da...,0.247567,1,0,0.335675
454,DATA 36,Data Scholars Seminar,0.243124,1,0,0.332343


### Hybrid Ranking Observations
The hybrid ranker produces more balanced recommendations than semantic similarity alone. It combines semantic relevance with subject preferences and a penalty for special-study courses, allowing highly relevant courses outside the preferred subject list to remain competitive while reducing weaker edge cases. 

### Comparing Hybrid vs. Semantic-only

In [33]:
semantic_top10 = semantic_recommendations[
    ["course", "title", "semantic_score"]
].head(10)

hybrid_top10 = hybrid_recommendations[
    ["course", "title", "final_score"]
].head(10)

display(semantic_top10)
display(hybrid_top10)

,course,title,semantic_score
1033,INDENG 142A,Introduction to Machine Learning and Data Anal...,0.493830
1973,UGBA 104,Introduction to Business Analytics,0.446975
596,ENGIN 183D,Product Management,0.430856
1917,STAT 159,Reproducible and Collaborative Statistical Dat...,0.411724
1908,STAT 133,Concepts in Computing with Data,0.357000
219,CDSS 170,Data Discovery Research Practicum,0.323936
453,DATA 188,Advanced Data Science Connector,0.315442
2002,UGBA 162,Brand Management and Strategy,0.304495
253,CHEM 98,Supervised Group Study,0.277914
434,STAT C8,Foundations of Data Science,0.273818


,course,title,final_score
1973,UGBA 104,Introduction to Business Analytics,0.485231
1917,STAT 159,Reproducible and Collaborative Statistical Dat...,0.458793
1908,STAT 133,Concepts in Computing with Data,0.417750
453,DATA 188,Advanced Data Science Connector,0.386582
2002,UGBA 162,Brand Management and Strategy,0.378371
1033,INDENG 142A,Introduction to Machine Learning and Data Anal...,0.370372
434,STAT C8,Foundations of Data Science,0.355364
458,DATA 94,Special Topics in Data Science,0.348362
464,DATA C6,Introduction to Computational Thinking with Da...,0.335675
454,DATA 36,Data Scholars Seminar,0.332343
